In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [2]:
df = pd.read_csv("merged_clean_data.csv")
df.head()

,customer_id,age,gender,item_purchased,category,purchase_amount_(usd),location,size,color,season,...,subscription_status,shipping_type,discount_applied,promo_code_used,previous_purchases,payment_method,frequency_of_purchases,state,avg_unemployment_rate,unemployment_group
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,...,Yes,Express,Yes,Yes,14,Venmo,Fortnightly,Kentucky,6.534397,High
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,...,Yes,Express,Yes,Yes,2,Cash,Fortnightly,Maine,5.631738,Medium
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,...,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly,Massachusetts,5.543794,Medium
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,...,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly,Rhode Island,6.385993,Medium
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,...,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually,Oregon,6.847163,High


In [3]:
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing values:\n", df.isna().sum())

Shape: (3900, 21)

Columns: ['customer_id', 'age', 'gender', 'item_purchased', 'category', 'purchase_amount_(usd)', 'location', 'size', 'color', 'season', 'review_rating', 'subscription_status', 'shipping_type', 'discount_applied', 'promo_code_used', 'previous_purchases', 'payment_method', 'frequency_of_purchases', 'state', 'avg_unemployment_rate', 'unemployment_group']

Missing values:
 customer_id               0
age                       0
gender                    0
item_purchased            0
category                  0
purchase_amount_(usd)     0
location                  0
size                      0
color                     0
season                    0
review_rating             0
subscription_status       0
shipping_type             0
discount_applied          0
promo_code_used           0
previous_purchases        0
payment_method            0
frequency_of_purchases    0
state                     0
avg_unemployment_rate     0
unemployment_group        0
dtype: int64


In [4]:
# feature engineering: PURCHASE FREQUENCY -> NUMERIC

freq_map = {
    "Daily": 30,
    "Weekly": 4,
    "Bi-Weekly": 2,
    "Fortnightly": 2,
    "Monthly": 1,
    "Quarterly": 0.33,
    "Semi-Annually": 0.16,
    "Annually": 0.083
}

df["purchase_frequency_numeric"] = df["frequency_of_purchases"].map(freq_map)

In [5]:
# encode binary targets (Yes/No -> 1/0)

binary_map = {"Yes": 1, "No": 0}

df["discount_applied_bin"] = df["discount_applied"].map(binary_map)
df["subscription_status_bin"] = df["subscription_status"].map(binary_map)

print(df[["discount_applied", "discount_applied_bin", "subscription_status", "subscription_status_bin"]].head())


  discount_applied  discount_applied_bin subscription_status  \
0              Yes                     1                 Yes   
1              Yes                     1                 Yes   
2              Yes                     1                 Yes   
3              Yes                     1                 Yes   
4              Yes                     1                 Yes   

   subscription_status_bin  
0                        1  
1                        1  
2                        1  
3                        1  
4                        1  


In [6]:
# select features for ML

feature_cols = [
    "avg_unemployment_rate",
    "purchase_amount_(usd)",
    "previous_purchases",
    "age",
    "purchase_frequency_numeric"
]

df_ml = df[feature_cols + ["discount_applied_bin", "subscription_status_bin"]].dropna()

print("ML dataset shape:", df_ml.shape)
df_ml.head()

ML dataset shape: (3316, 7)


,avg_unemployment_rate,purchase_amount_(usd),previous_purchases,age,purchase_frequency_numeric,discount_applied_bin,subscription_status_bin
0,6.534397,53,14,55,2.000,1,1
1,5.631738,64,2,19,2.000,1,1
2,5.543794,73,23,50,4.000,1,1
3,6.385993,90,49,21,4.000,1,1
4,6.847163,49,31,45,0.083,1,1


In [7]:
# train + evaluate

def train_and_evaluate(X, y, task_name="Task"):
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

    # Models
    models = {
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=1000, random_state=42))
        ]),
        "Random Forest": RandomForestClassifier(
            n_estimators=300,
            random_state=42
        )
    }

    print("\n" + "="*60)
    print(f"{task_name}")
    print("="*60)

    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)

        # ROC-AUC
        auc = None
        if hasattr(model, "predict_proba"):
            y_prob = model.predict_proba(X_test)[:, 1]
            auc = roc_auc_score(y_test, y_prob)

        print(f"\n--- {name} ---")
        print("Accuracy:", round(acc, 4))
        if auc is not None:
            print("ROC-AUC:", round(auc, 4))

        print("\nConfusion Matrix:")
        print(confusion_matrix(y_test, y_pred))

        print("\nClassification Report:")
        print(classification_report(y_test, y_pred, digits=4))

In [8]:
# TASK 1: discount applied (classification)

X1 = df_ml[feature_cols]
y1 = df_ml["discount_applied_bin"]

train_and_evaluate(X1, y1, task_name="Task 1: Predict Discount Applied (Yes/No)")


Task 1: Predict Discount Applied (Yes/No)

--- Logistic Regression ---
Accuracy: 0.5718
ROC-AUC: 0.486

Confusion Matrix:
[[474   0]
 [355   0]]

Classification Report:
              precision    recall  f1-score   support

           0     0.5718    1.0000    0.7276       474
           1     0.0000    0.0000    0.0000       355

    accuracy                         0.5718       829
   macro avg     0.2859    0.5000    0.3638       829
weighted avg     0.3269    0.5718    0.4160       829



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



--- Random Forest ---
Accuracy: 0.5428
ROC-AUC: 0.5113

Confusion Matrix:
[[340 134]
 [245 110]]

Classification Report:
              precision    recall  f1-score   support

           0     0.5812    0.7173    0.6421       474
           1     0.4508    0.3099    0.3673       355

    accuracy                         0.5428       829
   macro avg     0.5160    0.5136    0.5047       829
weighted avg     0.5254    0.5428    0.5244       829



In [9]:
# TASK 2: subscription status (classification)

X2 = df_ml[feature_cols]
y2 = df_ml["subscription_status_bin"]

train_and_evaluate(X2, y2, task_name="Task 2: Predict Subscription Status (Yes/No)")



Task 2: Predict Subscription Status (Yes/No)

--- Logistic Regression ---
Accuracy: 0.7286
ROC-AUC: 0.4789

Confusion Matrix:
[[604   0]
 [225   0]]

Classification Report:
              precision    recall  f1-score   support

           0     0.7286    1.0000    0.8430       604
           1     0.0000    0.0000    0.0000       225

    accuracy                         0.7286       829
   macro avg     0.3643    0.5000    0.4215       829
weighted avg     0.5308    0.7286    0.6142       829



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



--- Random Forest ---
Accuracy: 0.7189
ROC-AUC: 0.4817

Confusion Matrix:
[[588  16]
 [217   8]]

Classification Report:
              precision    recall  f1-score   support

           0     0.7304    0.9735    0.8346       604
           1     0.3333    0.0356    0.0643       225

    accuracy                         0.7189       829
   macro avg     0.5319    0.5045    0.4494       829
weighted avg     0.6227    0.7189    0.6255       829

